# Human-in-the-loop validation: deciding where a person needs to check

This builds directly on the evaluation template. Once you know how right a model is, per category, the next question is where a person still needs to look at its work before it reaches a real decision. Reviewing everything doesn't scale; reviewing nothing lets a quiet failure run for months before anyone notices. This notebook is about landing somewhere in between, deliberately, using the numbers evaluation gives you instead of a gut feeling.

**Time**: ~25 minutes
**Cost**: A few cents at most for the LLM-as-judge section (estimate first, per the token cost guide); the BLEU/ROUGE section runs entirely locally and costs nothing.

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../session_1/setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

## Recap: what the evaluation template already gave you

The evaluation template scored a classifier's predictions per category: precision (does it cry wolf), recall (does it miss real cases), and a confusion matrix showing which categories the model mixes up with which. That's the right tool for one specific kind of task — classification, where every item has one fixed, correct label to compare against.

Two things that notebook doesn't cover, and this one does:

- What do you do with a task that has no fixed right answer? A tagline, a subject line, a support reply draft — there's no single correct output sitting in a spreadsheet to check against, so accuracy-style metrics don't apply.
- Once you know a model's precision and recall, what do you actually *do* with that number? This is where human-in-the-loop validation comes in: deciding, category by category, whether a person needs to check the output before it reaches someone, or whether the model's track record is good enough to let it through unchecked.

## Two kinds of tasks, two kinds of metrics

Classification tasks — the five-category feedback classifier from the evaluation template — have a fixed answer key: every item has one correct label, so accuracy, precision, and recall all work by comparing a prediction against that fixed answer.

Generative tasks don't have that. Ask a model to write three subject lines for a promotion, and there's no single "correct" subject line to compare against — several very different outputs could all be equally good. Comparing generated text this way needs a different toolkit: **BLEU** and **ROUGE** when there's one reference text worth comparing against, and **LLM-as-judge** when there isn't, or when what actually matters — tone, brand voice, whether a claim is accurate — isn't something word-overlap can measure at all.

## BLEU and ROUGE: what they actually measure

Both compare a generated piece of text against one reference text you already trust, and both work by counting overlapping words and short phrases — **n-grams**, meaning sequences of *n* consecutive words — rather than understanding meaning.

**BLEU** (bilingual evaluation understudy) started in machine translation. It's precision-focused: of the words and phrases in the generated text, how many also appear in the reference? A generated sentence that repeats the reference almost word-for-word scores high; a fluent, accurate paraphrase that uses different words scores low, because BLEU has no way to know the two mean the same thing.

**ROUGE** (recall-oriented understudy for gisting evaluation) started in summarization. It's recall-focused: of the words and phrases in the reference, how many made it into the generated text? Same blind spot as BLEU, just measured from the other direction.

Install the one library the course hasn't used yet — `nltk` (for BLEU) is already in Colab; `rouge-score` isn't:

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = "the customer was refunded within three business days after the return was received".split()
exact_match = reference  # identical text, for comparison
paraphrase = "the refund was processed within three business days of the return arriving".split()

# SmoothingFunction avoids BLEU collapsing to 0 on short sentences with no shared 4-word
# sequences -- a known quirk of the raw score, not a sign the paraphrase is actually wrong.
smoothing = SmoothingFunction().method1

print(f"Exact match:  {sentence_bleu([reference], exact_match, smoothing_function=smoothing):.3f}")
print(f"Paraphrase:   {sentence_bleu([reference], paraphrase, smoothing_function=smoothing):.3f}")

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

reference_text = "The customer was refunded within three business days after the return was received."
close_paraphrase = "The refund was processed within three business days of the return arriving."
different_wording = "Refunds typically take a few days once we get the returned item."

for label, text in [("Close paraphrase", close_paraphrase), ("Same meaning, different wording", different_wording)]:
    scores = scorer.score(reference_text, text)
    print(f"{label}")
    print(f"  rouge1 F1: {scores['rouge1'].fmeasure:.3f}   rougeL F1: {scores['rougeL'].fmeasure:.3f}")

## Why these are a weak fit for most marketing generative work

Both demos above show the same failure: text that means the same thing, phrased differently, scores noticeably lower than a near-identical copy — even though a person reading both would call the paraphrase just as good, maybe better.

That's fine for tasks with one genuinely correct answer, like translating a sentence or summarizing a document against a reference summary someone wrote. It breaks down for marketing generation, where the whole point is usually that there isn't one right answer: three different taglines for the same brief can all be strong, share almost no words, and still all deserve a high score.

Know these two names — they show up constantly in papers and vendor benchmarks — but don't reach for them as a default metric on a generative marketing task. Reach for them when there's a genuine one-right-answer reference to compare against (translation, summarization against a source document), and reach for LLM-as-judge or human review everywhere else.

## LLM-as-judge: scoring generated text with a second model

The pattern: give a model the generated output, the brief or instructions it was written from, and a **rubric** — a short list of what "good" means for this task — and ask it to score against that rubric. It scales further than a person reviewing every item, and it can judge things BLEU and ROUGE can't touch, like tone or whether a claim in the copy is actually supported by the brief.

It comes with its own failure modes, worth knowing before trusting a judge score:

- **Verbosity bias** — judges tend to rate longer answers higher, independent of quality.
- **Position bias** — when comparing two outputs side by side, judges tend to favor whichever one comes first in the prompt.
- **Self-preference** — a model can rate output from its own model family more favorably than a genuinely neutral judge would.

None of these make LLM-as-judge useless, but they mean a judge needs validating against real human ratings on a sample before you trust its scores at scale — the same way you'd sanity-check a new hire's judgment before handing them the whole queue.

**Before running a batch job, estimate the cost** (habit from the token cost guide). A handful of judge calls on `gemini-2.5-flash-lite` costs a fraction of a cent.

In [ ]:
# Already installed above -- google-genai is in session_2/requirements.txt.
# This cell is intentionally left as a no-op if you're running cells out of order.
!pip install -q -r requirements.txt

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types
import json
import time

MODEL_NAME = "gemini-2.5-flash-lite"
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

JUDGE_RUBRIC = """Score the generated marketing copy on three dimensions, each from 1 (poor) to 5 (excellent):
- tone: does it match a warm, direct brand voice, with no hype and no jargon?
- clarity: is the call to action unambiguous?
- accuracy: does it avoid any claim that isn't supported by the brief?

Brief: "{brief}"
Generated copy: "{copy}"

Respond with only a JSON object, no other text, in exactly this shape:
{{"tone": <1-5>, "clarity": <1-5>, "accuracy": <1-5>, "reasoning": "<one sentence>"}}"""

def judge_copy(brief: str, copy: str, retries: int = 2) -> dict:
    prompt = JUDGE_RUBRIC.format(brief=brief, copy=copy)
    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0),
            )
            raw = response.text.strip()
            # Models sometimes wrap JSON in a code fence despite instructions -- strip it before parsing.
            if raw.startswith("```"):
                raw = raw.strip("`")
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            return json.loads(raw)
        except Exception as e:
            if attempt == retries:
                return {"tone": None, "clarity": None, "accuracy": None, "reasoning": f"judge failed: {e}"}
            time.sleep(2)

brief = "Announce free shipping on orders over $50, for environmentally conscious millennials."
draft = "Free shipping on orders $50+. Because saving the planet shouldn't cost extra to get your stuff."

result = judge_copy(brief, draft)
print(result)

## Deciding where a human checkpoint actually belongs

Metrics, whichever kind, only matter once they change what happens next. Here's the actual decision, applied to the five-category classifier from the evaluation template.

For each category, ask two questions. **Is this a high-stakes category** — one where a false negative (a miss) costs real money or a real customer, the way Returns & Refunds does if it feeds a churn-prevention workflow? And separately, **is the model's recall on this category good enough to trust without a second look?**

A category can fail either question on its own. A high-stakes category with excellent recall still deserves a checkpoint, because "usually right" isn't the same guarantee as "always right" when the cost of being wrong is high. A low-stakes category with poor recall doesn't need a person watching every single item, but it does need someone to notice before the miss rate quietly gets worse.

In [ ]:
def route_for_review(category: str, recall: float, high_stakes: set,
                      mandatory_below: float = 0.85, spot_check_below: float = 0.95) -> str:
    """
    Decide how a category's predictions should be reviewed, given its recall
    and whether it's flagged as high-stakes.

    mandatory_below / spot_check_below are thresholds you set deliberately, not
    defaults to trust blindly -- they should reflect what "good enough" actually
    means for your business, the same way choosing a threshold did in the
    evaluation template.
    """
    if category in high_stakes:
        return "mandatory review"
    if recall < mandatory_below:
        return "mandatory review"
    if recall < spot_check_below:
        return "spot-check sample"
    return "auto-approve"

# Recall figures a category-by-category run of the evaluation template might produce.
per_category_recall = {
    "Shipping & Delivery": 0.97,
    "Product Quality": 0.91,
    "Pricing & Billing": 0.80,
    "Customer Service": 0.96,
    "Returns & Refunds": 0.93,
}

# From the evaluation template: a miss here means a customer drops out of a
# churn-prevention workflow, so it stays under review regardless of recall.
HIGH_STAKES_CATEGORIES = {"Returns & Refunds"}

for category, recall in per_category_recall.items():
    decision = route_for_review(category, recall, HIGH_STAKES_CATEGORIES)
    print(f"{category:<22} recall={recall:.2f}  ->  {decision}")

## Sampling for everything that isn't mandatory

"Spot-check" and "auto-approve" aren't the same as "never look." A category with good recall today can drift — a model update, a new slang term, a seasonal shift in what customers write about — and nothing in a single evaluation run warns you that's happening. The fix isn't reviewing everything; it's a standing sample that keeps a person's eye on the categories you've decided not to check exhaustively.

A simple version: review 100% of anything routed to mandatory review, a fixed percentage of anything routed to spot-check, and log every disagreement between the model and the reviewer. If the disagreement rate on a spot-checked category starts climbing, that's the signal to re-run the evaluation template and reconsider the routing — not a signal to just increase the sample size and hope.

In [ ]:
import pandas as pd

# Standing in for a batch of freshly classified predictions -- in practice, this
# would be the `predictions` output from a run of the evaluation template.
predictions_df = pd.DataFrame({
    "id": range(1, 21),
    "category": (
        ["Shipping & Delivery"] * 4 + ["Product Quality"] * 4 + ["Pricing & Billing"] * 4
        + ["Customer Service"] * 4 + ["Returns & Refunds"] * 4
    ),
})

ROUTING = {
    category: route_for_review(category, recall, HIGH_STAKES_CATEGORIES)
    for category, recall in per_category_recall.items()
}
predictions_df["routing"] = predictions_df["category"].map(ROUTING)

SPOT_CHECK_RATE = 0.5  # review half of everything routed to spot-check

def build_review_queue(df: pd.DataFrame, spot_check_rate: float = SPOT_CHECK_RATE, seed: int = 1) -> pd.DataFrame:
    mandatory = df[df["routing"] == "mandatory review"]
    spot_check = df[df["routing"] == "spot-check sample"].sample(frac=spot_check_rate, random_state=seed)
    return pd.concat([mandatory, spot_check]).sort_values("id")

queue = build_review_queue(predictions_df)
print(f"Review queue: {len(queue)} of {len(predictions_df)} predictions ({len(queue) / len(predictions_df):.0%})")
queue

## Wrap-up

Two separate skills, both extending what the evaluation template started. For generative tasks with no fixed right answer, BLEU and ROUGE work only when there's a genuine reference text to compare against, and LLM-as-judge fills the gap everywhere else — once validated against real human ratings. For any task with metrics, per-category precision and recall, plus a plain judgment call about what's high-stakes, decide where a human checkpoint is mandatory, where a sample is enough, and where the model's track record earns it a pass.

Neither replaces judgment. The recall thresholds and the high-stakes list in this notebook are placeholders — the actual numbers are a business decision, made by the same people evaluation itself is for.

**Questions?** Post in the Circle community.